# Celeb-DF v2 and the DeepFakeDetection holdout

Cross-dataset evaluation on the official Celeb-DF v2 testing list, threshold recalibration, and the held-out DeepFakeDetection subset (Section 4.3, Tables 6 to 8).

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import pandas as pd
from pathlib import Path

BASE=Path('/content/drive/MyDrive/deepfake_finetuning'); OUT=BASE/'rebuild_2026'
CDF=BASE/'Celeb-DF-v2'; VIDEO_EXT={'.mp4','.avi','.mov','.mkv'}

lst=next((p for p in CDF.rglob('List_of_testing_videos.txt')), None)
print('list file:', lst)
if lst is None:
    print('MISSING - download List_of_testing_videos.txt from the Celeb-DF v2 release '
          'and drop it in the Celeb-DF-v2 folder. Without it Table 5 is not like-for-like.')
else:
    official={}
    for line in open(lst):
        line=line.strip()
        if line:
            lab,rel=line.split(None,1)
            official[rel.replace('\\','/').strip()]=int(lab)
    print(f'{len(official)} videos in the official test list')

    rows,missing=[],[]
    for rel,lab in official.items():
        p=CDF/rel
        if not p.exists():
            p=next((q for q in CDF.rglob(Path(rel).name)
                    if q.parent.name==Path(rel).parent.name), None)
        if p is None: missing.append(rel); continue
        is_real = 'synthesis' not in p.parent.name.lower()
        if lab != (1 if is_real else 0):
            print('label mismatch:', rel, lab)
        rows.append(dict(video_path=str(p), video_id=p.stem,
                         group_id=p.stem.split('_')[0],   # target identity
                         label=0 if is_real else 1,
                         method=p.parent.name, partition='test'))

    df=pd.DataFrame(rows)
    print(f'\n{len(df)} found, {len(missing)} missing')
    if missing: print('examples:', missing[:5])
    print(df.groupby(['method','label']).size())
    print('identities:', df.group_id.nunique())
    df.to_csv(OUT/'manifests'/'celebdf_manifest.csv', index=False)
    print('wrote celebdf_manifest.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
list file: /content/drive/MyDrive/deepfake_finetuning/Celeb-DF-v2/List_of_testing_videos.txt
518 videos in the official test list

518 found, 0 missing
method           label
Celeb-real       0        108
Celeb-synthesis  1        340
YouTube-real     0         70
dtype: int64
identities: 124
wrote celebdf_manifest.csv


In [ ]:
!pip -q install --no-deps facenet-pytorch==2.6.0
from google.colab import drive; drive.mount('/content/drive')
import cv2, numpy as np, pandas as pd, torch, time
from pathlib import Path
from PIL import Image
from facenet_pytorch import MTCNN

BASE=Path('/content/drive/MyDrive/deepfake_finetuning'); OUT=BASE/'rebuild_2026'
FR=OUT/'frames'/'celebdf'; FR.mkdir(parents=True,exist_ok=True)
K,SIZE=20,300
df=pd.read_csv(OUT/'manifests'/'celebdf_manifest.csv')
dev='cuda' if torch.cuda.is_available() else 'cpu'
det=MTCNN(image_size=SIZE,margin=20,keep_all=False,post_process=False,
          select_largest=True,device=dev)
print(dev,'|',len(df),'videos')

def faces(video,out_dir,uid):
    out_dir.mkdir(parents=True,exist_ok=True)
    cap=cv2.VideoCapture(str(video)); n=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    want=set(np.linspace(0,max(n-1,0),min(n,K)).round().astype(int).tolist()) if n else set()
    w,i=[],0
    while want and i<=max(want):
        ok,fr=cap.read()
        if not ok: break
        if i in want:
            img=Image.fromarray(cv2.cvtColor(fr,cv2.COLOR_BGR2RGB))
            try: f=det(img)
            except Exception: f=None
            if f is not None:
                a=f.permute(1,2,0).numpy(); a=((a-a.min())/(np.ptp(a)+1e-8)*255).astype('uint8')
                d=out_dir/f'{uid}__f{i:05d}.jpg'
                Image.fromarray(a).resize((SIZE,SIZE)).save(d,quality=95); w.append(str(d))
        i+=1
    cap.release(); return w

rows,failed,t0=[],[],time.time()
for n,r in enumerate(df.itertuples(),1):
    uid=f'{r.method}__{r.video_id}'
    d=FR/('fake' if r.label else 'real')
    done=sorted(d.glob(f'{uid}__f*.jpg'))
    try: paths=[str(p) for p in done] if done else faces(r.video_path,d,uid)
    except Exception as e: failed.append((uid,str(e))); paths=[]
    rows+=[dict(frame_path=p,video_id=uid,group_id=r.group_id,label=r.label,
                method=r.method,partition='test') for p in paths]
    if n%50==0:
        pd.DataFrame(rows).to_csv(OUT/'frames'/'celebdf_frames.csv',index=False)
        el=time.time()-t0
        print(f'{n}/{len(df)} | {len(rows)} frames | {el/60:.0f}m | '
              f'~{(el/n)*(len(df)-n)/60:.0f}m left',flush=True)

idx=pd.DataFrame(rows); idx.to_csv(OUT/'frames'/'celebdf_frames.csv',index=False)
assert idx.frame_path.is_unique
print(f'\n{len(idx)} frames from {idx.video_id.nunique()} videos, {len(failed)} failures')
print(idx.groupby(['method','label']).size())
print('detection rate %.1f%%'%(100*len(idx)/(K*idx.video_id.nunique())))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 72.9 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cuda | 518 videos
50/518 | 1000 frames | 3m | ~24m left
100/518 | 2000 frames | 5m | ~22m left
150/518 | 2998 frames | 8m | ~19m left
200/518 | 3997 frames | 10m | ~17m left
250/518 | 4997 frames | 13m | ~14m left
300/518 | 5997 frames | 15m | ~11m left
350/518 | 6997 frames | 18m | ~9m left
400/518 | 7997 frames | 20m | ~6m left
450/518 | 8997 frames | 23m | ~3m left
500/518 | 9997 frames | 25m | ~1m left

10357 frames from 518 videos, 0 failures
method           label
Celeb-real       0        2157
Celeb-synthesis  1        6800
YouTube-real     0        1400
dtype: int64
detection rate 100.0%


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import numpy as np, pandas as pd, torch, torch.nn as nn, time, json, joblib
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_v2_s
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score

OUT = Path('/content/drive/MyDrive/deepfake_finetuning/rebuild_2026')
RES, MOD, FEA = OUT/'results', OUT/'models', OUT/'features'
SIZE, CHUNK = 300, 4000
idx = pd.read_csv(OUT/'frames'/'celebdf_frames.csv')

TF = transforms.Compose([
    transforms.Resize((SIZE, SIZE)), transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406], [.229,.224,.225])])

class DS(Dataset):
    def __init__(s, p): s.p = p
    def __len__(s): return len(s.p)
    def __getitem__(s, i):
        for _ in range(3):
            try: return TF(Image.open(s.p[i]).convert('RGB'))
            except OSError: time.sleep(1)
        return torch.zeros(3, SIZE, SIZE)

dest = FEA/'celebdf_finetuned.npz'
if dest.exists():
    print('cached features')
    X = np.load(dest, allow_pickle=True)['X']
else:
    m = efficientnet_v2_s(weights=None)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 2)
    m.load_state_dict(torch.load(MOD/'efficientnetv2s_ffpp_clean.pth',
                                 map_location='cuda', weights_only=False)['model_state'])
    m = m.cuda().eval()
    P = idx.frame_path.tolist(); parts = []
    for c_i in range((len(P)+CHUNK-1)//CHUNK):
        dl = DataLoader(DS(P[c_i*CHUNK:(c_i+1)*CHUNK]), batch_size=64,
                        num_workers=2, pin_memory=True, timeout=180)
        F = []
        with torch.no_grad(), torch.amp.autocast('cuda'):
            for x in dl:
                F.append(m.avgpool(m.features(x.cuda())).flatten(1).float().cpu().numpy())
        parts.append(np.concatenate(F)); print(f'  chunk {c_i+1} ok', flush=True)
    X = np.concatenate(parts).astype('float32')
    np.savez_compressed(dest, X=X, y=idx.label.to_numpy(),
                        g=idx.group_id.to_numpy(), v=idx.video_id.to_numpy())

y, g, v = idx.label.to_numpy(), idx.group_id.to_numpy(), idx.video_id.to_numpy()
assert len(X) == len(y), 'feature/label mismatch'
print('features', X.shape)

b = joblib.load(MOD/'svm_ffpp_clean.pkl')
s = b['svm'].decision_function(b['scaler'].transform(X))

def M(y_, s_, thr, tag, v_=None):
    p = (s_ > thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_, p).ravel()
    r = dict(condition=tag, threshold=float(thr), auc=roc_auc_score(y_, s_),
             accuracy=accuracy_score(y_, p),
             balanced_accuracy=(tp/(tp+fn) + tn/(tn+fp))/2,
             fake_detection_rate=tp/(tp+fn), false_positive_rate=fp/(fp+tn),
             confusion_matrix=[[int(tn), int(fp)], [int(fn), int(tp)]])
    if v_ is not None:
        d = pd.DataFrame(dict(v=v_, y=y_, s=s_)).groupby('v').agg(y=('y','first'), s=('s','mean'))
        r['video_level_auc'] = roc_auc_score(d.y, d.s)
    print(f"{tag:34s} AUC {r['auc']:.4f} | FPR {r['false_positive_rate']:.4f} | "
          f"recall {r['fake_detection_rate']:.4f} | bal.acc {r['balanced_accuracy']:.4f}", flush=True)
    return r

out = {'zero_shot': M(y, s, 0, 'zero-shot (FF++ threshold)', v)}

rng = np.random.default_rng(42); ids = np.unique(g)
cal_ids = set(rng.choice(ids, size=max(2, int(len(ids)*0.2)), replace=False).tolist())
c = np.array([x in cal_ids for x in g]); e = ~c
assert not (set(g[c]) & set(g[e])), 'calibration/evaluation identity overlap'
print(f'\ncalibration {c.sum()} frames / {len(cal_ids)} identities | evaluation {e.sum()} frames')

grid = np.unique(np.quantile(s[c], np.linspace(.001, .999, 999)))
J = [((s[c] >= t)[y[c] == 1].mean() - (s[c] >= t)[y[c] == 0].mean()) for t in grid]
thr_j = float(grid[int(np.argmax(J))])
ok = [t for t in grid if (s[c] >= t)[y[c] == 0].mean() <= 0.10]
thr_f = float(min(ok)) if ok else thr_j

out['recalibrated'] = dict(
    calibration_frames=int(c.sum()),
    calibration_identities=len(cal_ids),
    evaluation_frames=int(e.sum()),
    default=M(y[e], s[e], 0, '  default threshold (eval subset)'),
    youden=M(y[e], s[e], thr_j, '  recalibrated (Youden J)'),
    fpr10=M(y[e], s[e], thr_f, '  recalibrated (FPR<=10% budget)'),
)

h = json.load(open(RES/'ffpp_indistribution_final.json'))
out['generalisation_gap'] = dict(
    ffpp_auc=h['auc'],
    celebdf_auc=out['zero_shot']['auc'],
    drop_percentage_points=100*(h['auc'] - out['zero_shot']['auc']),
)
print(f"\nFF++ {h['auc']:.4f} -> Celeb-DF {out['zero_shot']['auc']:.4f} = "
      f"{out['generalisation_gap']['drop_percentage_points']:.2f} pp drop")

np.savez_compressed(RES/'celebdf_test_scores.npz', y=y, scores=s, groups=g, videos=v)
json.dump(out, open(RES/'celebdf_crossdataset.json','w'), indent=2, default=float)
print('saved')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  chunk 1 ok
  chunk 2 ok
  chunk 3 ok
features (10357, 1280)
zero-shot (FF++ threshold)         AUC 0.6589 | FPR 0.6556 | recall 0.8174 | bal.acc 0.5809

calibration 2220 frames / 24 identities | evaluation 8137 frames
  default threshold (eval subset)  AUC 0.6655 | FPR 0.6676 | recall 0.8292 | bal.acc 0.5808
  recalibrated (Youden J)          AUC 0.6655 | FPR 0.3490 | recall 0.5975 | bal.acc 0.6243
  recalibrated (FPR<=10% budget)   AUC 0.6655 | FPR 0.1087 | recall 0.2876 | bal.acc 0.5894

FF++ 0.9848 -> Celeb-DF 0.6589 = 32.59 pp drop
saved


In [ ]:
!pip -q install --no-deps facenet-pytorch==2.6.0
from google.colab import drive; drive.mount('/content/drive')
import cv2, numpy as np, pandas as pd, torch, torch.nn as nn, time, json, joblib
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_v2_s
from facenet_pytorch import MTCNN
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score

BASE=Path('/content/drive/MyDrive/deepfake_finetuning'); OUT=BASE/'rebuild_2026'
RES,MOD,FEA=OUT/'results',OUT/'models',OUT/'features'
FR=OUT/'frames'/'dfd'; FR.mkdir(parents=True,exist_ok=True)
K,SIZE,CHUNK=20,300,4000

man=pd.read_csv(OUT/'manifests'/'ffpp_manifest.csv')
dfd=man[man.partition=='dfd_holdout'].reset_index(drop=True)
print(f'{len(dfd)} DFD videos, {dfd.group_id.nunique()} actor groups')

dev='cuda'
det=MTCNN(image_size=SIZE,margin=20,keep_all=False,post_process=False,
          select_largest=True,device=dev)
def faces(video,out_dir,uid):
    out_dir.mkdir(parents=True,exist_ok=True)
    cap=cv2.VideoCapture(str(video)); n=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    want=set(np.linspace(0,max(n-1,0),min(n,K)).round().astype(int).tolist()) if n else set()
    w,i=[],0
    while want and i<=max(want):
        ok,fr=cap.read()
        if not ok: break
        if i in want:
            img=Image.fromarray(cv2.cvtColor(fr,cv2.COLOR_BGR2RGB))
            try: f=det(img)
            except Exception: f=None
            if f is not None:
                a=f.permute(1,2,0).numpy(); a=((a-a.min())/(np.ptp(a)+1e-8)*255).astype('uint8')
                d=out_dir/f'{uid}__f{i:05d}.jpg'
                Image.fromarray(a).resize((SIZE,SIZE)).save(d,quality=95); w.append(str(d))
        i+=1
    cap.release(); return w

rows,t0=[],time.time()
for n,r in enumerate(dfd.itertuples(),1):
    uid=f'DFD__{r.video_id}'; d=FR/'fake'
    done=sorted(d.glob(f'{uid}__f*.jpg'))
    paths=[str(p) for p in done] if done else faces(r.video_path,d,uid)
    rows+=[dict(frame_path=p,video_id=uid,group_id=r.group_id,label=1,
                method='DeepFakeDetection') for p in paths]
    if n%50==0: print(f'{n}/{len(dfd)} | {len(rows)} frames | {(time.time()-t0)/60:.0f}m',flush=True)
fake=pd.DataFrame(rows)

# pair with the 140 real videos of the FF++ official test partition (also unseen)
ff=pd.read_csv(OUT/'frames'/'ffpp_frames.csv')
real=ff[(ff.partition=='test')&(ff.label==0)][['frame_path','video_id','group_id','label','method']]
idx=pd.concat([real,fake],ignore_index=True)
idx.to_csv(OUT/'frames'/'dfd_frames.csv',index=False)
print(f'\n{len(idx)} frames | real {len(real)} fake {len(fake)} | '
      f'detection rate {100*len(fake)/(K*fake.video_id.nunique()):.1f}%')

TF=transforms.Compose([transforms.Resize((SIZE,SIZE)),transforms.ToTensor(),
   transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
class DS(Dataset):
    def __init__(s,p): s.p=p
    def __len__(s): return len(s.p)
    def __getitem__(s,i):
        for _ in range(3):
            try: return TF(Image.open(s.p[i]).convert('RGB'))
            except OSError: time.sleep(1)
        return torch.zeros(3,SIZE,SIZE)

dest=FEA/'dfd_finetuned.npz'
if dest.exists(): X=np.load(dest,allow_pickle=True)['X']
else:
    m=efficientnet_v2_s(weights=None); m.classifier[1]=nn.Linear(m.classifier[1].in_features,2)
    m.load_state_dict(torch.load(MOD/'efficientnetv2s_ffpp_clean.pth',
                                 map_location='cuda',weights_only=False)['model_state'])
    m=m.cuda().eval(); P=idx.frame_path.tolist(); parts=[]
    for ci in range((len(P)+CHUNK-1)//CHUNK):
        dl=DataLoader(DS(P[ci*CHUNK:(ci+1)*CHUNK]),batch_size=64,num_workers=2,
                      pin_memory=True,timeout=180); F=[]
        with torch.no_grad(), torch.amp.autocast('cuda'):
            for x in dl: F.append(m.avgpool(m.features(x.cuda())).flatten(1).float().cpu().numpy())
        parts.append(np.concatenate(F)); print(f'  chunk {ci+1} ok',flush=True)
    X=np.concatenate(parts).astype('float32')
    np.savez_compressed(dest,X=X,y=idx.label.to_numpy(),v=idx.video_id.to_numpy())

y,v=idx.label.to_numpy(),idx.video_id.to_numpy()
b=joblib.load(MOD/'svm_ffpp_clean.pkl'); s=b['svm'].decision_function(b['scaler'].transform(X))
p=(s>0).astype(int); tn,fp,fn,tp=confusion_matrix(y,p).ravel()
vid=pd.DataFrame(dict(v=v,y=y,s=s)).groupby('v').agg(y=('y','first'),s=('s','mean'))
r=dict(condition='DeepFakeDetection (unseen manipulation)',auc=roc_auc_score(y,s),
       video_level_auc=roc_auc_score(vid.y,vid.s),accuracy=accuracy_score(y,p),
       balanced_accuracy=(tp/(tp+fn)+tn/(tn+fp))/2,fake_detection_rate=tp/(tp+fn),
       false_positive_rate=fp/(fp+tn),confusion_matrix=[[int(tn),int(fp)],[int(fn),int(tp)]],
       n_fake_videos=int(fake.video_id.nunique()),n_real_videos=int(real.video_id.nunique()),
       note='DFD fakes vs the real videos of the FF++ official test partition')
for k,val in r.items(): print(f'  {k:22s} {val}')
np.savez_compressed(RES/'dfd_test_scores.npz',y=y,scores=s,videos=v)
json.dump(r,open(RES/'dfd_holdout.json','w'),indent=2,default=float)
print('saved')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
433 DFD videos, 112 actor groups
50/433 | 996 frames | 5m
100/433 | 1994 frames | 9m
150/433 | 2985 frames | 13m
200/433 | 3983 frames | 17m
250/433 | 4976 frames | 21m
300/433 | 5957 frames | 26m
350/433 | 6956 frames | 29m
400/433 | 7953 frames | 34m

11409 frames | real 2799 fake 8610 | detection rate 99.4%
  chunk 1 ok
  chunk 2 ok
  chunk 3 ok
  condition              DeepFakeDetection (unseen manipulation)
  auc                    0.9688521161738949
  video_level_auc        0.9897723523589574
  accuracy               0.9091068454728723
  balanced_accuracy      0.9095179587533129
  fake_detection_rate    0.9087108013937282
  false_positive_rate    0.08967488388710254
  confusion_matrix       [[2548, 251], [786, 7824]]
  n_fake_videos          433
  n_real_videos          140
  note                   DFD fakes vs the real videos of the FF++ official test 